In [ ]:
!pip install nltk scikit-learn pandas

import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

import pandas as pd
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df=pd.read_csv('/content/drive/MyDrive/Data/nlp_dataset.csv')
df

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear
...,...,...
5932,i begun to feel distressed for you,fear
5933,i left feeling annoyed and angry thinking that...,anger
5934,i were to ever get married i d have everything...,joy
5935,i feel reluctant in applying there because i w...,fear


In [ ]:
df.head()

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


In [ ]:
df.isnull().sum()

,0
Comment,0
Emotion,0


In [ ]:
df['Emotion'].value_counts()

,count
Emotion,
anger,2000
joy,2000
fear,1937


In [ ]:
def preprocess(text):
  text = text.lower()
  text = re.sub(r'[^a-zA-Z\s]', '', text)
  return text

df['cleaned_Text'] = df['Comment'].apply(preprocess)
df['cleaned_Text']

,cleaned_Text
0,i seriously hate one subject to death but now ...
1,im so full of life i feel appalled
2,i sit here to write i start to dig out my feel...
3,ive been really angry with r and i feel like a...
4,i feel suspicious if there is no one outside l...
...,...
5932,i begun to feel distressed for you
5933,i left feeling annoyed and angry thinking that...
5934,i were to ever get married i d have everything...
5935,i feel reluctant in applying there because i w...


In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
  tokens = word_tokenize(text)
  tokens = [word for word in tokens if word not in stop_words]
  return tokens

df['processed_text'] = df['cleaned_Text'].apply(remove_stopwords)
df['processed_text']

,processed_text
0,"[seriously, hate, one, subject, death, feel, r..."
1,"[im, full, life, feel, appalled]"
2,"[sit, write, start, dig, feelings, think, afra..."
3,"[ive, really, angry, r, feel, like, idiot, tru..."
4,"[feel, suspicious, one, outside, like, rapture..."
...,...
5932,"[begun, feel, distressed]"
5933,"[left, feeling, annoyed, angry, thinking, cent..."
5934,"[ever, get, married, everything, ready, offer,..."
5935,"[feel, reluctant, applying, want, able, find, ..."


1. **Lowercasing**:
All text was converted to lowercase. This makes words such as Happy and happy treated as the same word and reduces unnecessary features.
2. **Removing** **special** **characters**:
Punctuation and unnecessary characters were removed to reduce noise in the text.
3. **Tokenization**:
Each sentence was divided into individual words. For example, "I am very happy" becomes ["I", "am", "very", "happy"].
4. **Stop**-**word** **removal**:
Common words such as the, is, am, a, and of were removed because they usually provide less useful information for classification.

**Impact** **on** **Model** **Performance**

These preprocessing steps reduce noise and make the text more consistent. This helps TF-IDF create more meaningful numerical features and can improve the efficiency and classification performance of Naive Bayes and SVM models.

In [ ]:
vectorizer = TfidfVectorizer(analyzer=lambda x: x)
X = vectorizer.fit_transform(df['processed_text'])
y = df['Emotion']
print(X.shape)

(5937, 8832)


TF-IDF transforms text data into numerical feature vectors by assigning weights to words based on their importance in each document and across the entire dataset. This allows machine-learning algorithms to process and classify text data.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train,X_test,y_train,y_test

(<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 43097 stored elements and shape (4749, 8832)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 10843 stored elements and shape (1188, 8832)>,
 4945      joy
 5428     fear
 1344    anger
 1888    anger
 2480     fear
         ...  
 3772    anger
 5191     fear
 5226    anger
 5390     fear
 860      fear
 Name: Emotion, Length: 4749, dtype: object,
 1867    anger
 3988      joy
 4516     fear
 1397    anger
 1669    anger
         ...  
 642       joy
 1253     fear
 3094      joy
 3733     fear
 3523      joy
 Name: Emotion, Length: 1188, dtype: object)

In [ ]:
#Naive bayes

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
nb_pred

array(['anger', 'joy', 'fear', ..., 'anger', 'fear', 'joy'], dtype='<U5')

In [ ]:
#SVM

svm_model = SVC()
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)
svm_pred

array(['anger', 'joy', 'fear', ..., 'joy', 'fear', 'joy'], dtype=object)

In [ ]:
#model evaluation

nb_accuracy = accuracy_score(y_test, nb_pred)
nb_f1 = f1_score(y_test, nb_pred, average='weighted')

svm_accuracy = accuracy_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred, average='weighted')

print("Naive Bayes Accuracy:", nb_accuracy, "F1-Score:", nb_f1)
print("SVM Accuracy:", svm_accuracy, "F1-Score:", svm_f1)

Naive Bayes Accuracy: 0.9116161616161617 F1-Score: 0.9115095144506908
SVM Accuracy: 0.936026936026936 F1-Score: 0.935993621606674


1. Naive Bayes (NB)
* Brief explanation:

Naive Bayes is a supervised classification algorithm that uses the probability of words to predict the class of a text.

* Suitability for emotion classification:

It is suitable because it works well with text data and TF-IDF features. It can learn the association between words and emotions such as anger, fear, joy, and sadness. It is also simple and fast.

2. Support Vector Machine (SVM)

*Brief explanation:

SVM is a supervised classification algorithm that finds a decision boundary to separate different classes.

*Suitability for emotion classification:

SVM works well with high-dimensional TF-IDF features and can effectively separate text into different emotion categories.